# 50. MPS로 오버레이 영상 한 편을 완성하는 시간

검증 영상 c04 116장을 기존 분석 명령으로 처리합니다. 원판 베이스라인과 원본 YOLO26n-pose는 긴 변 640, 배치 1, FP32로 실행하고, 원본 크기로 H.264 영상을 저장합니다. 별도 예열 없이 한 편을 실행합니다. 프로그램을 새로 시작한 시간과 모델 로딩부터 결과 파일을 쓴 시간, 두 값을 나눠 봅니다. 49번의 프레임별 추론·후처리 측정에는 영상 저장이 빠져 있었습니다.

In [ ]:
import json
import os
from pathlib import Path
import subprocess
import sys
from time import perf_counter

import cv2
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "wlcv/config.py").is_file())
video = "c04_sn_f_01"
command = [sys.executable, "-m", "wlcv.analyze", video, "--device", "mps"]
env = {**os.environ, "PYTORCH_ENABLE_MPS_FALLBACK": "0"}
start = perf_counter()
result = subprocess.run(command, cwd=root, env=env, stdout=subprocess.PIPE, text=True, check=True)
command_seconds = perf_counter() - start
print(result.stdout.strip())
folder = Path(next(line.removeprefix("로컬 결과: ") for line in result.stdout.splitlines()
                   if line.startswith("로컬 결과: ")))
summary = json.loads((folder / "summary.json").read_text())
timing = json.loads((folder / "timing.json").read_text())
frames = pd.read_csv(folder / "frames.csv")
print(f"명령 시작부터 종료까지 {command_seconds:.2f}초")

## 저장한 영상을 처음부터 끝까지 확인

MP4를 다시 열어 실제 디코딩된 장수와 출력 크기, 재생 fps를 확인합니다. 이 추가 검수는 위의 측정에서 제외합니다. 이륙·무릎 통과·마지막 프레임은 완성된 영상에서 꺼낸 그림입니다.

In [ ]:
lift = summary["manual_phase_interval"]["lift_off_frame"]
knee = summary["manual_phase_interval"]["knee_pass_frame"]
expected = len(frames)
assert summary["device"] == "mps" and expected == summary["frame_count"] == 116
assert frames.frame_index.tolist() == list(range(expected))
capture = cv2.VideoCapture(str(folder / "overlay.mp4"))
samples = {}
count = 0
try:
    fps = capture.get(cv2.CAP_PROP_FPS)
    while True:
        ok, image = capture.read()
        if not ok:
            break
        assert image.shape[:2] == (1920, 1080)
        if count in (lift, knee, expected - 1):
            samples[count] = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        count += 1
finally:
    capture.release()
assert count == expected and abs(fps - summary["overlay_fps"]) < 0.01
seconds = count / fps
print(f"{count}장 · 1080×1920 · 재생 {fps:.2f} fps · 영상 길이 {seconds:.3f}초")
fig, axes = plt.subplots(1, 3, figsize=(8, 4.6), layout="constrained")
for ax, (index, rgb) in zip(axes, samples.items()):
    ax.imshow(rgb)
    ax.set_title(f"Frame {index}")
    ax.axis("off")
plt.show()

## 어디에 시간이 걸렸나?

기존 명령은 첫 번째 디코딩에서 원판을 잡고 궤적을 연결한 후, 영상을 다시 읽어 자세를 추론하고 오버레이를 씁니다. 두 번째 읽기와 자세 모델 준비 등 미분리 시간은 render_other에 포함됩니다. writer는 프레임 쓰기와 인코더 종료 대기를 합친 값입니다. 라이브러리 import와 프로세스 종료는 명령 전체 시간에 포함됩니다.

In [ ]:
pipeline = timing["pipeline_seconds"]
stages = pd.Series(timing["stage_seconds"], name="seconds")
display(stages.round(3).to_frame())
print(f"분석 함수 {pipeline:.2f}초 ({count/pipeline:.2f} fps)")
print(f"명령 전체 {command_seconds:.2f}초 ({count/command_seconds:.2f} fps)")
print(f"재생 시간 대비 명령 소요시간 {command_seconds/seconds:.2f}배")
ax = stages.sort_values().plot.barh(title="MPS overlay / 116 frames")
ax.set_xlabel("Wall time (s)")
ax.figure.tight_layout()
ax.figure.savefig(folder / "stage_times.png", dpi=130)
plt.show()
report = dict(command_seconds=command_seconds, command_fps=count/command_seconds,
              decoded_frames=count, playback_fps=fps, playback_seconds=seconds)
(folder / "playback_check.json").write_text(json.dumps(report, indent=2))
display(frames.loc[frames.frame_index.isin([lift, knee]),
                   ["frame_index", "plate_status", "plate_cy_px", "trunk_angle_deg"]])
print(summary["plate_counts"], "상승량(px)", summary["manual_phase_interval"]["rise_px"],
      "체간 변화(°)", summary["trunk_change_deg"])